# Session 4: How a floor plan becomes a number

**259.075 Data-integrated Algorithmic Design Processes II** · Session 4 of 7

For three sessions the inputs were numbers somebody picked: compactness, U-value. Every one of
those numbers is a decision about what the model is allowed to see, and compactness in
particular throws away almost everything. A courtyard block and a slab with the same perimeter
and the same area are the same building as far as session 1 is concerned.

Today the input is the outline itself. Nobody hands you a number for a shape, so the first
question is how to make one, and the second is what you lose in the making.

In [ ]:
import sys
import numpy as np
import torch
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

sys.path.insert(0, "../data")
from autoencoder import Autoencoder            # the class; the weights come from file

bilder = np.load("../data/grundrisse.npy")     # 1071 outlines, 32 x 32, from raster.py
tabelle = np.genfromtxt("../data/grundrisse.csv", delimiter=",", names=True,
                        dtype=None, encoding="utf-8")
print(bilder.shape, "outlines")
print("fill ratio on average:", round(float(bilder.mean()), 3))

## Three ways into a number

The same building, three times.

As two numbers: area and perimeter, or the compactness we built out of them. Two numbers keep
the size and a rough sense of how compact the shape is; the arrangement of corners, projections
and courtyards is gone.

As a picture: the outline dropped on a grid of 32 by 32 cells, one number per cell, 1024 in
total. Nothing is interpreted; a cell is inside or it is not.

As a sequence: go around the outline and write down each edge as a length and a turn. That is
the description an architect would give, it keeps the corners exactly, and it has a different
length for every building, which is exactly why the rest of this session uses the grid instead.

> Every representation is a decision about what the model may notice. There is no neutral one.

In [ ]:
import json as _json
from raster import ringe, meter, ausrichten, rastere
import raster

geo = _json.loads(open("../data/gebaeude_wien.geojson", encoding="utf-8").read())
haus = 59
bild = bilder[haus]

merkmal = next(m for m in geo["features"]
               if str(m["properties"]["id"]) == str(tabelle["id"][haus]))
lat0 = float(np.asarray(next(ringe(merkmal["geometry"]))[0])[:, 1].mean())
teil = next(ringe(merkmal["geometry"]))
ringe_m = [meter(r, lat0, teil[0][0]) for r in teil]
dreh, (mitte, rundheit) = ausrichten(ringe_m)
umriss = [(r - mitte) @ dreh.T for r in ringe_m]

fig = make_subplots(rows=1, cols=3, subplot_titles=(
    "the outline itself", "1024 numbers: the grid", "where the edges are"))
for ring in umriss:
    fig.add_trace(go.Scatter(x=ring[:, 0], y=ring[:, 1], mode="lines",
                             line=dict(color="#4c78a8"), showlegend=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=bild[::-1], colorscale="Greys", reversescale=True,
                         showscale=False), row=1, col=2)
kanten = np.abs(np.diff(bild, axis=0, prepend=0)) + np.abs(np.diff(bild, axis=1, prepend=0))
fig.add_trace(go.Heatmap(z=kanten[::-1], colorscale="Oranges", showscale=False), row=1, col=3)
fig.update_layout(height=340, title=(
    f"building {tabelle['id'][haus]}, {tabelle['bauperiode'][haus]}, "
    f"{float(tabelle['flaeche_m2'][haus]):.0f} m2, compactness "
    f"{float(tabelle['kompaktheit'][haus]):.2f}"))
fig.update_yaxes(scaleanchor="x", row=1, col=1)
fig.update_yaxes(scaleanchor="x2", row=1, col=2)
fig.update_yaxes(scaleanchor="x3", row=1, col=3)
fig.show()

## Putting a shape on a grid

Two decisions hide in that middle picture, and both matter more than the resolution.

The outline has to be centred and turned before it is rasterised, otherwise the model learns
where a building stands in Vienna rather than what shape it has. `data/raster.py` puts the
centre of area at the origin, turns the principal axis horizontal, and scales uniformly with a
margin, so that a long building stays long and a square one stays square.

The turning has a failure mode worth knowing. For a nearly square outline the principal axis is
barely defined, and a small change in the outline can swing it by ninety degrees. The script
records how round each building is, and ninety of the 1071 are round enough that their
orientation should not be trusted.

> Alignment is not preprocessing. It decides what the model is able to learn at all.

The second decision is the resolution. The building above is a Gründerzeit block with three
courtyards. Compare the four resolutions and find the one at which a courtyard stops being a
courtyard.

In [ ]:
aufloesungen = [8, 16, 32, 64]
stapel = {}
for n in aufloesungen:
    raster.N = n
    stapel[n] = rastere(ringe_m, dreh, mitte)
raster.N = 32

fig = make_subplots(rows=1, cols=4, subplot_titles=[f"{n} x {n}" for n in aufloesungen])
for spalte, n in enumerate(aufloesungen, start=1):
    fig.add_trace(go.Heatmap(z=stapel[n][::-1], colorscale="Greys", reversescale=True,
                             showscale=False), row=1, col=spalte)
    fig.update_yaxes(scaleanchor=f"x{spalte}", row=1, col=spalte)
fig.update_layout(height=300, title=f"the same outline at four resolutions, "
                                    f"roundness {rundheit:.2f}")
fig.show()

## The bottleneck

A thousand and twenty-four numbers per building is not a description, it is the building again.
What we want is a handful of numbers that carry the shape, and the way to get them is to force a
model through a narrow gap.

An *autoencoder* is two networks back to back. The *encoder* takes 1024 numbers down to a few,
the *decoder* takes those few back up to 1024, and the whole thing is trained to reproduce its
own input. The narrow part in the middle is the *bottleneck*, and it is the only reason the
model has to learn anything: without it, copying the input through would be a perfect solution
and nothing would be learned.

Three models were trained the same way, with 32, 8 and 2 numbers in the middle. They are
separate models, not one model with a dial. You cannot narrow a bottleneck at runtime.

> The bottleneck is what forces a decision about what matters. Remove it and the model learns
> to be a wire.

In [ ]:
modelle = {}
for enge in (32, 8, 2):
    m = Autoencoder(enge)
    m.load_state_dict(torch.load(f"../data/ae_{enge}.pt"))
    m.eval()
    modelle[enge] = m

X = torch.tensor(bilder.reshape(len(bilder), -1), dtype=torch.float32)
gezeigt = [3, 12, 44, 101, 260]

fig = make_subplots(rows=4, cols=len(gezeigt), vertical_spacing=0.04,
                    row_titles=["original", "32 numbers", "8 numbers", "2 numbers"])
for spalte, j in enumerate(gezeigt, start=1):
    fig.add_trace(go.Heatmap(z=bilder[j], colorscale="Greys", reversescale=True,
                             showscale=False), row=1, col=spalte)
    for zeile, enge in enumerate((32, 8, 2), start=2):
        with torch.no_grad():
            rueck = torch.sigmoid(modelle[enge](X[j:j + 1])).numpy().reshape(32, 32)
        fig.add_trace(go.Heatmap(z=rueck, colorscale="Greys", reversescale=True,
                                 showscale=False), row=zeile, col=spalte)
fig.update_layout(height=760, title="the same five buildings through three bottlenecks")
fig.update_yaxes(autorange="reversed")
fig.show()

In [ ]:
for enge in (32, 8, 2):
    with torch.no_grad():
        rueck = torch.sigmoid(modelle[enge](X)).numpy().reshape(-1, 32, 32)
    treffer = float(np.mean((rueck > 0.5) == (bilder > 0.5)))
    # a model that says "empty" everywhere already gets this many cells right:
    print(f"{enge:2d} numbers: {treffer:.3f} of cells correct")
print(f"all-empty baseline: {1 - float(bilder.mean()):.3f}")

Look at the pictures before you look at the numbers, because the numbers are misleading. All
three models get around ninety-five percent of the cells right or better, and a model that simply said
"empty" everywhere would get seventy. The interesting part is not the percentage, it is which
kind of information goes first.

What do you see? Say it out loud before reading on. The usual guesses are corners and
courtyards, and at least one of them is wrong.

The honest description of these five: corners round off first, small projections go with them,
and what survives down to two numbers is roughly length and width. That is not nothing. Most of
Vienna's building stock is a long rectangle, and two numbers describe a long rectangle rather
well, which is why the two-number model looks better than it has any right to.

## The map of shapes

Every building now has eight numbers. Eight is too many to draw, so we project to two.

The way to think about it: a shadow of an object is two-dimensional and you still recognise the
object, as long as the light comes from a sensible direction. *Principal component analysis*
picks that direction, the one that keeps as much of the spread as possible. No formula here; the
cell prints how much of the spread survives, and that number is the honest measure of what the
picture is worth.

In [ ]:
daten = np.load("../data/codes.npz")
codes = daten["code_8"]

zentriert = codes - codes.mean(axis=0)
U, S, Vt = np.linalg.svd(zentriert, full_matrices=False)
karte = zentriert @ Vt[:2].T
anteil = float((S[:2] ** 2).sum() / (S ** 2).sum())

print(f"the two axes of the map keep {anteil:.1%} of the spread in the eight numbers")

fig = px.scatter(x=karte[:, 0], y=karte[:, 1], color=tabelle["startjahr"].astype(float),
                 hover_name=tabelle["id"].astype(str),
                 hover_data={"period": tabelle["bauperiode"], "area": tabelle["flaeche_m2"],
                             "roundness": tabelle["rundheit"]},
                 color_continuous_scale="Turbo", height=560,
                 labels={"x": "first axis", "y": "second axis", "color": "period from"})
fig.update_traces(marker=dict(size=6, opacity=0.7))
fig.update_layout(title="1071 Vienna outlines, laid out by shape")
fig.show()

## Does the map mean anything?

A code in which nearness does not mean similarity is a decoration. So we check instead of
promising. The check runs on all eight numbers; the map above is only their shadow on two axes,
and two buildings close on the map need not be close in the code.

Write the two functions below. The first gives the indices of the nearest codes to a given one;
the second walks from one code to another. Both are one or two lines, and both get used for the
rest of the session.

In [ ]:
def naechste(code, alle, n=4):
    """Indices of the n rows of `alle` closest to `code`, nearest first."""

    # YOUR TURN: two lines. Distances from `code` to every row of `alle`, then the indices of
    # the n smallest, closest first. np.argsort is allowed and is the point.
    raise NotImplementedError("nearest neighbours missing")



def zwischen(a, b, t):
    """The point t of the way from code a to code b."""

    # YOUR TURN: one line. The point t of the way from a to b, with t = 0 at a and t = 1 at b.
    raise NotImplementedError("interpolation missing")

In [ ]:
zufall = np.random.default_rng(4)
proben = zufall.choice(len(codes), 5, replace=False)

fig = make_subplots(rows=5, cols=4, vertical_spacing=0.03,
                    column_titles=["the building", "nearest", "second", "third"])
for zeile, j in enumerate(proben, start=1):
    for spalte, nachbar in enumerate(naechste(codes[j], codes, 4), start=1):
        fig.add_trace(go.Heatmap(z=bilder[nachbar], colorscale="Greys", reversescale=True,
                                 showscale=False), row=zeile, col=spalte)
fig.update_layout(height=900, title="five buildings and their nearest neighbours in the eight numbers")
fig.update_yaxes(autorange="reversed")
fig.show()

Judge it yourselves. If the neighbours look alike, the code is doing its job. If they do not, say
so and ask why: maybe eight numbers are too few, maybe the alignment failed on round outlines,
maybe a thousand buildings are not enough to fill an eight-dimensional space. All three are real
answers, and "this does not work" is a result, not a failure.

## Walking between two buildings

The last step is the first one in which something appears that was never in the data. Take two
codes, walk from one to the other, and decode every point along the way.

In [ ]:
von, nach = 12, 260
schritte = np.linspace(0, 1, 9)
with torch.no_grad():
    reihe = [torch.sigmoid(modelle[8].decoder(
        torch.tensor(zwischen(codes[von], codes[nach], float(t)),
                     dtype=torch.float32).unsqueeze(0))).numpy().reshape(32, 32)
        for t in schritte]

fig = make_subplots(rows=1, cols=len(schritte),
                    subplot_titles=[f"{t:.2f}" for t in schritte])
for spalte, bild_i in enumerate(reihe, start=1):
    fig.add_trace(go.Heatmap(z=bild_i, colorscale="Greys", reversescale=True, showscale=False),
                  row=1, col=spalte)
fig.update_layout(height=260, title="from one Vienna building to another, through the code")
fig.update_yaxes(autorange="reversed")
fig.show()

None of the shapes in the middle is a building in Vienna. They are what the decoder makes of
points between two codes, and whether they are any good is a question you can only answer by
looking.

Try a pair of your own. Pick two ids from the map above, find their rows, and run the cell
again. Some pairs walk smoothly, others produce grey mush halfway across, and that is not a bug
in your code: the model was never asked what lies between two buildings. Session 5 is about
repairing exactly that.

## Tests

This cell has to run green before you submit.

In [ ]:
probe = np.array([[0.0, 0.0], [3.0, 0.0], [0.0, 1.0], [-2.0, 0.0], [0.0, 0.5]])
i = naechste(np.array([0.0, 0.0]), probe, 4)
assert list(i[:1]) == [0], f"the point itself has to come first, got {list(i)}"
assert list(i) == [0, 4, 2, 3], f"wrong order: {list(i)}"

gross = np.arange(60, dtype=float).reshape(20, 3)
ziel = gross[7] + 0.01
assert list(naechste(ziel, gross, 3)) == [7, 8, 6], "order wrong on a larger example"

a, b = np.array([0.0, 10.0]), np.array([4.0, -2.0])
assert np.allclose(zwischen(a, b, 0.0), a), "t = 0 has to give a"
assert np.allclose(zwischen(a, b, 1.0), b), "t = 1 has to give b"
assert np.allclose(zwischen(a, b, 0.25), [1.0, 7.0]), "quarter of the way is wrong"

mitte = zwischen(codes[von], codes[nach], 0.5)
with torch.no_grad():
    bild_mitte = torch.sigmoid(modelle[8].decoder(
        torch.tensor(mitte, dtype=torch.float32).unsqueeze(0))).numpy()
assert bild_mitte.shape == (1, 1024), "the decoder wants a code of length 8"
assert 0.02 < float(bild_mitte.mean()) < 0.9, (
    f"the midpoint decodes to something empty or solid: {float(bild_mitte.mean()):.3f}")

print("all green")

## Where you stand now

You can name three ways to turn a shape into numbers and say what each one discards. You know
that centring and turning an outline is not housekeeping but the decision about what the model
can learn. You can read a bottleneck: what survives it, what does not, and why a high percentage
of correct cells can mean very little. And you have a map in which nearness is supposed to mean
similarity, together with the habit of checking whether it does.

Next session we try to stand at an arbitrary point in that map and ask for a building. The
autoencoder will refuse, in an interesting way.

---

### Terms to learn

| term | in one sentence |
|---|---|
| representation | the form in which data reaches the model; always a choice, never neutral |
| rasterisation | putting a shape on a grid, one number per cell |
| alignment | centring, turning and scaling before the model sees anything |
| autoencoder | encoder and decoder trained to reproduce the input through a narrow middle |
| bottleneck | the few numbers in the middle; what forces the model to choose what matters |
| code, latent vector | the few numbers a particular input is compressed into |
| latent space | the space those codes live in; nearness is supposed to mean similarity |
| reconstruction | what comes back out of the decoder |
| principal component analysis | projecting many numbers to few while keeping as much spread as possible |
| interpolation | walking between two codes and decoding the points on the way |

## Homework

Ten points, due before the next session: four for running it, six for your thoughts on it.

**How to do it**

1. Fill in the `YOUR TURN` gap above, yourself or from the solution notebook. If you took it from
   the solution, say so in one line at the top of your answers; that costs nothing.
2. Below, change only the lines under `# YOUR CHOICE`.
3. Write your answers into the last cell.
4. Choose Kernel, Restart Kernel and Run All Cells, check that everything runs, and upload the
   notebook in TUWEL.

**How it is graded**

- Four points for the run: it runs without an error (one), it shows what the task asks for (one),
  and every number you give about your run appears in your output or is worked out from it in a
  step you write down (two). Numbers the questions give you, examples from outside the notebook
  and forecasts you mark as your own expectation are exempt.
- Six points for your answers, one per question, no halves. Say what you saw and what it means,
  and you get the point, also with borrowed code. A sentence that would fit any model gets none.

### The task

1. Pick a building from the table, a row from 0 to 1070, one you can describe in words. Send it
   through all three bottlenecks and keep the figure.
2. Enter an outline of your own as a list of corners in metres: your house, a design of yours,
   a building you know. The example in the cell is only there to show the format; replace it,
   since a submission with the example unchanged misses what the task asks you to show.
   The cell checks the outline, puts it through the same alignment and the 8-number model, and
   shows its ten nearest Vienna buildings.
3. Decide whether those ten would be any use for a design. "They are useless, and this is why"
   is a complete answer.

In [ ]:
# YOUR CHOICE: a row of the table, 0 to 1070
mein_haus = None

assert mein_haus is not None and 0 <= mein_haus <= 1070, "enter a row between 0 and 1070"
fig = make_subplots(rows=1, cols=4, subplot_titles=("original", "32 numbers", "8 numbers",
                                                    "2 numbers"))
fig.add_trace(go.Heatmap(z=bilder[mein_haus], colorscale="Greys", reversescale=True,
                         showscale=False), row=1, col=1)
for spalte, enge in enumerate((32, 8, 2), start=2):
    with torch.no_grad():
        rueck = torch.sigmoid(modelle[enge](X[mein_haus:mein_haus + 1])).numpy().reshape(32, 32)
    falsch = int(((rueck > 0.5) != (bilder[mein_haus] > 0.5)).sum())
    print(f"{enge:2d} numbers: {falsch} of 1024 cells wrong")
    fig.add_trace(go.Heatmap(z=rueck, colorscale="Greys", reversescale=True, showscale=False),
                  row=1, col=spalte)
fig.update_layout(height=300, title=(
    f"building {tabelle['id'][mein_haus]}, {tabelle['bauperiode'][mein_haus]}, "
    f"{float(tabelle['flaeche_m2'][mein_haus]):.0f} m2, "
    f"roundness {float(tabelle['rundheit'][mein_haus]):.2f}"))
fig.update_yaxes(autorange="reversed")
for name in fig.layout:
    if name.startswith("yaxis"):
        fig.layout[name].scaleanchor = "x" + name[5:]
fig.show()
print(f"filled cells in the original: {int((bilder[mein_haus] > 0.5).sum())}; "
      f"a model answering 'empty' everywhere would get exactly these wrong")

In [ ]:
# YOUR CHOICE: your own outline, corners in metres, going round once; the last corner joins
# the first by itself. The L below only shows the format; replace it.
aussen = np.array([[0, 0], [24, 0], [24, 8], [10, 8], [10, 20], [0, 20]], dtype=float)
eigene_ringe = [aussen]
# a courtyard is a second ring in the same coordinates, lying inside the first, e.g.
# hof = np.array([[3, 3], [7, 3], [7, 6], [3, 6]], dtype=float)
# eigene_ringe = [aussen, hof]


def beruehrt(p1, p2, p3, p4):
    # do the two segments share any point at all, touching and overlapping included?
    d = lambda a, b, c: (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])
    auf = lambda a, b, c: (min(a[0], b[0]) <= c[0] <= max(a[0], b[0])
                           and min(a[1], b[1]) <= c[1] <= max(a[1], b[1]))
    d1, d2, d3, d4 = d(p3, p4, p1), d(p3, p4, p2), d(p1, p2, p3), d(p1, p2, p4)
    if d1 * d2 < 0 and d3 * d4 < 0:
        return True
    return ((d1 == 0 and auf(p3, p4, p1)) or (d2 == 0 and auf(p3, p4, p2))
            or (d3 == 0 and auf(p1, p2, p3)) or (d4 == 0 and auf(p1, p2, p4)))


sauber = []
for r in eigene_ringe:
    r = np.asarray(r, dtype=float)
    assert r.ndim == 2 and r.shape[1] == 2, "each ring is a list of [x, y] corners"
    assert np.isfinite(r).all(), "a corner is not a number"
    if len(r) > 3 and np.allclose(r[0], r[-1], rtol=0, atol=1e-6):
        r = r[:-1]                                  # a repeated closing corner is not needed
    assert len(r) >= 3, "each ring needs at least three corners"
    sauber.append(r)
eigene_ringe = sauber

kanten = [(n, i, (r[i], r[(i + 1) % len(r)])) for n, r in enumerate(eigene_ringe)
          for i in range(len(r))]
for a in range(len(kanten)):
    na, ia, (p, q) = kanten[a]
    assert not np.allclose(p, q, rtol=0, atol=1e-6), "two corners in a row are the same point"
    for b in range(a + 1, len(kanten)):
        nb, ib, (s, t) = kanten[b]
        m = len(eigene_ringe[na])
        if na == nb and (ib - ia) % m in (1, m - 1):
            # neighbours share a corner; they must not fold back onto each other
            u, v = (q - p, t - s) if (ib - ia) % m == 1 else (t - s, q - p)
            assert not (u[0] * v[1] - u[1] * v[0] == 0 and u @ v < 0), (
                "the outline doubles back on itself; check the order of the corners")
            continue
        assert not beruehrt(p, q, s, t), (
            "edges cross or touch; check the order of the corners, and keep courtyards clear "
            "of each other and of the outer wall")
for hof in eigene_ringe[1:]:
    assert raster.im_polygon(hof, [eigene_ringe[0]]).all(), "a courtyard has to lie inside the outer ring"
    for anderer in eigene_ringe[1:]:
        if anderer is not hof:
            assert not raster.im_polygon(hof[:1], [anderer]).any(), "one courtyard lies inside another"

flaeche = abs(raster.flaeche_und_schwerpunkt(eigene_ringe[0])[0]) - sum(
    abs(raster.flaeche_und_schwerpunkt(h)[0]) for h in eigene_ringe[1:])
dreh_e, zusatz_e = ausrichten(eigene_ringe)
assert dreh_e is not None, "the outline is too thin or too small to align; check the corners"
mitte_e, rund_e = zusatz_e
mein_bild = rastere(eigene_ringe, dreh_e, mitte_e)
assert mein_bild is not None and mein_bild.sum() >= 10, (
    "on the 32 x 32 grid your outline covers almost nothing; it is too thin for this grid")

with torch.no_grad():
    mein_code = modelle[8].encoder(torch.tensor(mein_bild.reshape(1, -1))).numpy()[0]
nachbarn = naechste(mein_code, codes, 10)

print(f"your outline: {flaeche:.0f} m2, {int(mein_bild.sum())} of 1024 cells filled, "
      f"roundness {rund_e:.2f}" + ("  (above 0.8: the turning is not to be trusted)"
                                   if rund_e > 0.8 else ""))
fig = make_subplots(rows=2, cols=6, vertical_spacing=0.12, horizontal_spacing=0.02,
                    subplot_titles=["yours"] + [f"{n}: {tabelle['id'][j]}"
                                                for n, j in enumerate(nachbarn, start=1)])
fig.add_trace(go.Heatmap(z=mein_bild, colorscale="Oranges", showscale=False), row=1, col=1)
for n, j in enumerate(nachbarn, start=1):
    r, c = divmod(n, 6)
    fig.add_trace(go.Heatmap(z=bilder[j], colorscale="Greys", reversescale=True,
                             showscale=False), row=r + 1, col=c + 1)
fig.update_layout(height=440, title="your outline and its ten nearest Vienna buildings")
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
for name in fig.layout:
    if name.startswith("yaxis"):
        fig.layout[name].scaleanchor = "x" + name[5:]
fig.show()

for n, j in enumerate(nachbarn, start=1):
    print(f"{n:2d}  {tabelle['id'][j]}  {tabelle['bauperiode'][j]:>10}  "
          f"{float(tabelle['flaeche_m2'][j]):7.0f} m2  distance "
          f"{float(np.linalg.norm(codes[j] - mein_code)):.2f}")

### Your interpretation

1. **Size.** How many cells does each bottleneck get wrong on your building, and how does that compare with the number a model answering "empty" everywhere would get wrong?
2. **Plausibility.** What changes in your building's reconstruction as the bottleneck narrows? Point to the picture. If nothing visible is lost, or a narrower bottleneck does better, say so; that happens and is a valid observation.
3. **Units.** Your own outline has a size in square metres, printed above. What size does it have once it is on the grid, and what does that mean for the areas listed next to your ten neighbours?
4. **Range.** Say in one sentence where your outline comes from. Is it the kind of shape the model was trained on, Vienna buildings mostly from before 1918? Say what the encoder does with it either way.
5. **The neighbours.** Name one neighbour, by its number, that does not fit, and give a possible reason: the alignment, the grid, the bottleneck, or simply that Vienna has no such building. If all ten fit, say what they share.
6. **Design consequence.** Would the ten neighbours be any use to you for a design? Say for what, or why not.

*Your text here.*